# Agreement between raters & fairness across groups

_Metrics & Evaluation — measuring models, data & statistics_

**Two questions, one idea — do two judges agree more than chance, and does the model treat each group the same?**

Both families ask the same shaped question: here is a gap; is it bigger than it ought to be?

       Agreement compares two raters on the same items. The naive score is plain agreement — the fraction of items they labeled the same. But some agreement happens by pure luck, so we subtract off the luck. That gives a chance-corrected score: 1 = perfect agreement, 0 = no better than coin-flips, negative = worse than chance.

---

This notebook is a practice scaffold for the **Agreement between raters & fairness across groups** lesson. Run the cells, then tackle the practice problems at the bottom. _Save a copy to your Drive (File → Save a copy in Drive) to edit and keep your work._

In [ ]:
# Setup — numpy / pandas / scikit-learn / matplotlib ship with Colab.
!pip install -q fairlearn
import numpy as np, matplotlib.pyplot as plt

## First, look at the data

Before training on it, see what each example actually contains. Each row is one example; the columns are its features, plus a class label.

In [ ]:
from sklearn.datasets import load_breast_cancer

data = load_breast_cancer(as_frame=True)
print("rows x columns:", data.frame.shape)
print("feature columns:", list(data.data.columns))
print("target classes :", list(data.target_names))
data.frame.head()

## Reference implementation — scikit-learn

In [ ]:
import numpy as np
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.metrics import cohen_kappa_score

# ---------- FAMILY 1: inter-rater agreement ----------
# two raters labeling the same 8 items (1 = tumor, 0 = clear)
rater_a = [1, 1, 0, 0, 1, 0, 1, 0]
rater_b = [1, 0, 0, 0, 1, 0, 1, 1]
print("Cohen's kappa        :", round(cohen_kappa_score(rater_a, rater_b), 3))
# weighted kappa for ORDERED categories (0 < 1 < 2): off-by-one hurts less
sev_a = [0, 1, 2, 2, 1, 0, 1, 2]
sev_b = [0, 2, 2, 1, 1, 0, 0, 2]
print("Quadratic-weighted k :",
      round(cohen_kappa_score(sev_a, sev_b, weights="quadratic"), 3))

# ICC (Intraclass Correlation Coefficient) for NUMERIC ratings:
#   import pingouin as pg
#   pg.intraclass_corr(data=long_df, targets="item", raters="judge", ratings="score")
# Krippendorff's alpha (any data type, tolerates missing) -- from scratch sketch:
#   alpha = 1 - Do / De   with Do = observed disagreement, De = expected disagreement.

# ---------- FAMILY 2: fairness across groups ----------
from fairlearn.metrics import (MetricFrame, selection_rate,
    demographic_parity_difference, equalized_odds_difference)
from sklearn.metrics import recall_score, precision_score

data = load_breast_cancer()
X = data.data
y = (data.target == 0).astype(int)            # 1 = malignant (positive)
mean_radius = X[:, 0]                          # the feature we bin into groups

Xtr, Xte, ytr, yte, rtr, rte = train_test_split(
    X, y, mean_radius, test_size=0.4, random_state=0, stratify=y)
clf = make_pipeline(StandardScaler(),
                    LogisticRegression(max_iter=5000)).fit(Xtr, ytr)
y_pred = clf.predict(Xte)

# protected GROUP = mean-radius tertiles (small / medium / large tumors)
q1, q2 = np.quantile(rte, [1/3, 2/3])
group = np.where(rte <= q1, "small",
        np.where(rte <= q2, "medium", "large"))

# per-group rates in one table
mf = MetricFrame(
    metrics={"selection_rate": selection_rate,
             "TPR (recall)":  recall_score,
             "precision":     precision_score},
    y_true=yte, y_pred=y_pred, sensitive_features=group)
print(mf.by_group)                            # selection / TPR / precision per group
print("demographic parity diff:",
      round(demographic_parity_difference(yte, y_pred, sensitive_features=group), 3))
print("equalized odds diff    :",
      round(equalized_odds_difference(yte, y_pred, sensitive_features=group), 3))

## Visualize the data & results

_Train a real classifier, bin a real feature into a protected group, and look: does the model select each group at the same rate? The gap is the unfairness._

In [ ]:
import numpy as np
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline

data = load_breast_cancer()
X = data.data
y = (data.target == 0).astype(int)            # 1 = malignant (positive)
mean_radius = X[:, 0]

Xtr, Xte, ytr, yte, rtr, rte = train_test_split(
    X, y, mean_radius, test_size=0.4, random_state=0, stratify=y)
clf = make_pipeline(StandardScaler(),
                    LogisticRegression(max_iter=5000)).fit(Xtr, ytr)
y_pred = clf.predict(Xte)

# protected GROUP = mean-radius tertiles on the test set
q1, q2 = np.quantile(rte, [1/3, 2/3])
group = np.where(rte <= q1, "small",
        np.where(rte <= q2, "medium", "large"))

for g in ["small", "medium", "large"]:
    m = group == g
    sel = y_pred[m].mean()                     # selection rate
    print(g, "n=", m.sum(), "selection_rate=%.3f" % sel)
# -> small 0.013, medium 0.197, large 0.842
print("demographic-parity diff:", round(0.842 - 0.013, 3))   # 0.829
print("disparate-impact ratio :", round(0.013 / 0.842, 3))   # 0.016 (fails 80% rule)

## Practice

Try each one in the empty cell below it, then reveal the worked solution.

**Problem 1.** Two annotators label 100 comments as "toxic" or "fine" and agree on 92 of them. You report "92% agreement." Your lead asks for Cohen's kappa instead. Toxic is rare: annotator A calls "toxic" 8% of the time, B 10%. Roughly what is kappa, and why is it so different from 92%?

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- Write the observed agreement. — _They matched on 92 of 100, so p_o = 0.92._
- Compute chance agreement p_e. — _Both say "toxic" with prob 0.08 x 0.10 = 0.008, both say "fine" with 0.92 x 0.90 = 0.828; p_e = 0.836._
- Plug into kappa. — _kappa = (0.92 - 0.836)/(1 - 0.836) = 0.084/0.164 ≈ 0.51._

**Answer:** $p_o = 0.92$, but because "toxic" is rare, chance agreement is already $p_e = (0.08)(0.10) + (0.92)(0.90) = 0.836$. So $\kappa = \frac{0.92 - 0.836}{1 - 0.836} \approx 0.51$ — only "moderate" agreement, not the 92% the raw number suggested. This is the prevalence effect: when one category dominates, most agreement is luck, and kappa strips that away. Report kappa with the raw agreement and the rare-class rate so the picture is honest.

</details>

**Problem 2.** A hiring model approves 48% of group A and 36% of group B. (a) What is the demographic-parity difference? (b) What is the disparate-impact ratio, and does it pass the 80% rule? (c) The model also misses qualified people more often in group B. Which fairness metric should you report for that harm, and why?

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- Take the selection-rate gap. — _Demographic-parity difference = |0.48 - 0.36| = 0.12._
- Take the ratio of the lower to higher rate. — _Disparate impact = 0.36 / 0.48 = 0.75, which is below the 0.8 threshold._
- Match the metric to the "miss" harm. — _Missing qualified people is a false negative; equal opportunity equalizes the true-positive rate (recall) across groups._

**Answer:** (a) Demographic-parity difference $= |0.48 - 0.36| = 0.12$. (b) Disparate-impact ratio $= 0.36/0.48 = 0.75$, which is below 0.8 — it fails the 80% rule, flagging that group B is selected materially less often. (c) Since the harm is missing qualified people (false negatives), report equal opportunity — the gap in true-positive rate (recall) between the groups — because that metric directly measures whether the model catches the deserving "yes" cases equally in each group.

</details>

**Problem 3.** A reviewer insists your risk model must simultaneously satisfy demographic parity, equalized odds, AND predictive parity across two groups whose true base rates differ (12% positive vs 30% positive). Is this achievable, and what should you tell them?

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- Note the base rates differ. — _The two groups genuinely contain positives at different rates, which is the trigger condition for the impossibility result._
- Recall the impossibility theorem. — _With unequal base rates and an imperfect model, equalizing precision and equalizing TPR/FPR cannot both hold — fixing one gap opens another._
- Convert it to a choice. — _Fairness becomes a decision about which harm to equalize, not a checklist you satisfy in full._

**Answer:** No — it is provably impossible. When the groups have different true base rates (12% vs 30%) and the model is not perfect, the impossibility theorem says predictive parity (equal precision) and equalized odds (equal TPR and FPR) cannot both hold; calibration adds a third conflicting demand. Tell the reviewer that fairness here is a trade-off, not a checkbox: pick the single definition matching the real-world harm (equal opportunity if misses hurt most, predictive parity if false "yes" calls hurt most), report the others as context, and document the choice.

</details>